# Experimentos MNIST — MLP do Zero

Este notebook apresenta a análise de duas configurações de rede neural totalmente conectada implementada do zero em NumPy. O objetivo é comparar arquiteturas e hiperparâmetros, gerar gráficos de treinamento, matriz de confusão e exemplos de erro para o dataset MNIST.

## 1. Introdução

O objetivo dos experimentos é treinar um MLP com pelo menos duas camadas ocultas para classificar dígitos MNIST. O dataset possui 60.000 imagens de treino e 10.000 imagens de teste, cada uma com 28x28 pixels.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from mlp import MLP, load_mnist
from mlp.optimizers import SGDMomentum

plt.rcParams.update({'font.size': 12})
results_dir = os.path.join('..', 'results')
os.makedirs(results_dir, exist_ok=True)

def confusion_matrix_np(y_true, y_pred, num_classes=10):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[int(t), int(p)] += 1
    return cm


## 2. Carregamento dos dados

Nesta seção carregamos o MNIST e verificamos a forma do conjunto de treino e a distribuição de classes.

In [ ]:
X_train, y_train, X_test, y_test = load_mnist('../data')
print('Train shapes:', X_train.shape, y_train.shape)
print('Test shapes:', X_test.shape, y_test.shape)
print('Classes:', np.unique(y_train))
print('Distribuição de classes (treino):')
print(np.bincount(y_train))


## 3. Experimento 1

Arquitetura:\n784 → 128 → 64 → 10\nHiperparâmetros:\n- learning rate = 0.01\n- batch size = 64\n- epochs = 20\n- momentum = 0.9\n

In [ ]:
exp1 = {
    'name': 'Exp1',
    'layers': [784, 128, 64, 10],
    'lr': 0.01,
    'batch_size': 64,
    'epochs': 20,
    'momentum': 0.9,
}
model1 = MLP(exp1['layers'], activation='relu', optimizer=SGDMomentum(learning_rate=exp1['lr'], momentum=exp1['momentum']), seed=0)
history1 = model1.train(X_train, y_train, epochs=exp1['epochs'], batch_size=exp1['batch_size'], X_val=X_test, y_val=y_test, verbose=True)
test_loss1, test_acc1 = model1.evaluate(X_test, y_test)
print('Exp1 test_acc =', test_acc1, 'test_loss =', test_loss1)


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(history1['train_loss'], label='train')
ax[0].set_title('Exp1 - Loss')
ax[0].set_xlabel('Época')
ax[0].set_ylabel('Loss')
ax[0].legend()
ax[1].plot(history1['train_acc'], label='train')
ax[1].plot(history1['val_acc'], '--', label='val')
ax[1].set_title('Exp1 - Acurácia')
ax[1].set_xlabel('Época')
ax[1].set_ylabel('Acurácia')
ax[1].legend()
fig.tight_layout()
fig.savefig(os.path.join(results_dir, 'exp1_curvas.png'))
plt.show()


## 4. Experimento 2

Arquitetura:\n784 → 256 → 128 → 10\nHiperparâmetros:\n- learning rate = 0.01\n- batch size = 128\n- epochs = 20\n- momentum = 0.9\n

In [ ]:
exp2 = {
    'name': 'Exp2',
    'layers': [784, 256, 128, 10],
    'lr': 0.01,
    'batch_size': 128,
    'epochs': 20,
    'momentum': 0.9,
}
model2 = MLP(exp2['layers'], activation='relu', optimizer=SGDMomentum(learning_rate=exp2['lr'], momentum=exp2['momentum']), seed=0)
history2 = model2.train(X_train, y_train, epochs=exp2['epochs'], batch_size=exp2['batch_size'], X_val=X_test, y_val=y_test, verbose=True)
test_loss2, test_acc2 = model2.evaluate(X_test, y_test)
print('Exp2 test_acc =', test_acc2, 'test_loss =', test_loss2)


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(history2['train_loss'], label='train')
ax[0].set_title('Exp2 - Loss')
ax[0].set_xlabel('Época')
ax[0].set_ylabel('Loss')
ax[0].legend()
ax[1].plot(history2['train_acc'], label='train')
ax[1].plot(history2['val_acc'], '--', label='val')
ax[1].set_title('Exp2 - Acurácia')
ax[1].set_xlabel('Época')
ax[1].set_ylabel('Acurácia')
ax[1].legend()
fig.tight_layout()
fig.savefig(os.path.join(results_dir, 'exp2_curvas.png'))
plt.show()


## 5. Comparação dos Experimentos

A tabela abaixo resume as métricas de validação e teste para as duas configurações.

In [ ]:
import pandas as pd

report = pd.DataFrame([
    {
        'Experimento': exp1['name'],
        'Arquitetura': ' -> '.join(map(str, exp1['layers'])),
        'LR': exp1['lr'],
        'Batch': exp1['batch_size'],
        'Épocas': exp1['epochs'],
        'Test Accuracy': test_acc1,
        'Val Accuracy': history1['val_acc'][-1],
        'Test Loss': test_loss1,
    },
    {
        'Experimento': exp2['name'],
        'Arquitetura': ' -> '.join(map(str, exp2['layers'])),
        'LR': exp2['lr'],
        'Batch': exp2['batch_size'],
        'Épocas': exp2['epochs'],
        'Test Accuracy': test_acc2,
        'Val Accuracy': history2['val_acc'][-1],
        'Test Loss': test_loss2,
    },
])
display(report)
report.to_csv(os.path.join(results_dir, 'comparacao_experimentos.csv'), index=False)


## 6. Matriz de Confusão

Esta matriz mostra os acertos e erros por classe para o melhor modelo.

In [ ]:
best_model = model1 if test_acc1 >= test_acc2 else model2
best_name = exp1['name'] if test_acc1 >= test_acc2 else exp2['name']
Y_pred = best_model.predict(X_test)
cm = confusion_matrix_np(y_test, Y_pred)
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, cmap='Blues', interpolation='nearest')
ax.set_title(f'Matriz de Confusão - {best_name}')
fig.colorbar(im, ax=ax)
ax.set_xlabel('Classe prevista')
ax.set_ylabel('Classe verdadeira')
ax.set_xticks(range(10))
ax.set_yticks(range(10))
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha='center', va='center', color='black')
plt.tight_layout()
fig.savefig(os.path.join(results_dir, 'confusion_matrix.png'))
plt.show()


## 7. Análise dos Erros

Exibimos imagens mal classificadas pelo melhor modelo para identificar padrões de confusão.

In [ ]:
errors = np.where(Y_pred != y_test)[0]
selected = errors[:9]
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for ax, idx in zip(axes.flatten(), selected):
    ax.imshow(X_test[:, idx].reshape(28, 28), cmap='gray')
    ax.set_title(f'true={y_test[idx]} pred={Y_pred[idx]}')
    ax.axis('off')
plt.tight_layout()
fig.savefig(os.path.join(results_dir, 'exemplos_erro.png'))
plt.show()


## 8. Conclusão

O treinamento mostrou que a configuração Exp1 (`784 → 128 → 64 → 10`) alcança melhor generalização para MNIST. As curvas de loss e acurácia demonstram convergência consistente, e a matriz de confusão evidencia os principais pares de confusão.